# Analisis Eksploratif

t-SNE/UMAP (volatilitas tinggi vs rendah), feature importance, dan analisis error (tugas 3c).

In [1]:
from c_common import *

D, text = load_all()
print({s: len(d) for s, d in D.items()}, text.shape)

{'train': 825, 'val': 176, 'test': 178} (1202, 5)


## 1. Bangun ulang Model C (C2_finbert_art_cls_HGB)

Fitur dibangun sama persis seperti `06_ablation` (FinBERT judul+lead per artikel, pooling [CLS], rata-rata harian, PCA 20 yang di-fit di hari train, ditambah `has`).
Model final dilatih ulang di train+val dengan hyperparameter tersimpan, lalu probabilitas test dibandingkan dengan `results_c/preds/`.
Kalau selisihnya tidak nol, angka di bawah TIDAK boleh dipakai (cek versi scikit-learn, hasil tersimpan dibuat dengan 1.8.0).

In [2]:
import sklearn
from sklearn.decomposition import PCA
from scipy.stats import spearmanr, binomtest

print("scikit-learn", sklearn.__version__)
ANALYSIS = RESULTS_C / "analysis"
ANALYSIS.mkdir(exist_ok=True)

C_NAME = "C2_finbert_art_cls_HGB"
TITLE, LEAD = "judul_gabungan", "lead_paragraph_gabungan"
day_text = (text[TITLE].fillna("") + ". " + text[LEAD].fillna("")).str.strip(" .")
has_news = (day_text.str.len() > 0).values

emb = pd.DataFrame(np.load(FEATURES_C / "c2_finbert_art_cls.npy"), index=text.index)
tr = D["train"]["date"]
tr = tr[has_news[text.index.get_indexer(tr)]]          # PCA hanya dari hari train yang ada beritanya
pca = PCA(n_components=20, random_state=SEED).fit(emb.loc[tr].values)
z = pca.transform(emb.values)
z[~has_news] = 0.0
comp = pd.DataFrame(z, index=text.index, columns=[f"c2_art_cls_{i}" for i in range(20)])
comp["c2_art_cls_has"] = has_news.astype(float)
for s in D:
    D[s] = D[s].merge(comp, left_on="date", right_index=True, how="left")
TXT = list(comp.columns)
COLS = PRICE + TXT

rec = json.loads((RESULTS_C / f"{C_NAME}.json").read_text())
p = rec["best_params"]
pv = make_model("HGB", p).fit(D["train"][COLS], D["train"]["target"]).predict_proba(D["val"][COLS])[:, 1]
sc_val = metrics(D["val"]["target"], (pv >= 0.5).astype(int), pv)
print("selisih maks skor val vs results_c:", max(abs(sc_val[k] - rec["val"][k]) for k in sc_val))

trval = pd.concat([D["train"], D["val"]])
model_c = make_model("HGB", p).fit(trval[COLS], trval["target"])
p_test = model_c.predict_proba(D["test"][COLS])[:, 1]
saved = pd.read_csv(RESULTS_C / "preds" / f"{C_NAME}_test.csv")
print("selisih maks prob test vs results_c/preds:", np.abs(p_test - saved["score"].values).max())
sc_test = metrics(D["test"]["target"], (p_test >= 0.5).astype(int), p_test)
print("test:", {k: round(v, 3) for k, v in sc_test.items()}, "| tersimpan:", {k: round(v, 3) for k, v in rec["test"].items()})

scikit-learn 1.8.0
selisih maks skor val vs results_c: 0.0


selisih maks prob test vs results_c/preds: 1.1102230246251565e-16
test: {'DA': 0.584, 'BalAcc': 0.561, 'F1_macro': 0.56, 'MCC': 0.127, 'AUC': 0.594} | tersimpan: {'DA': 0.584, 'BalAcc': 0.561, 'F1_macro': 0.56, 'MCC': 0.127, 'AUC': 0.594}


## 2. Analisis error Model A vs Model C per tanggal

Model A = `HGB [Harga]` dari `data/results/predictions.csv` (harga saja, prediksinya sama di kedua korpus, lihat `06_ablation`).
Model C = probabilitas test di atas. Keduanya dinilai pada 178 hari test yang sama.
Interval kepercayaan memakai *moving-block bootstrap* (blok 5 hari, 2.000 ulangan, seed 42) seperti Tugas 2.

In [3]:
P = pd.read_csv(DATA / "results" / "predictions.csv", parse_dates=["date"])
P = P[P.split == "test"].reset_index(drop=True)
te = D["test"].reset_index(drop=True).copy()
assert (P["date"].values == te["date"].values).all() and (P["target"].values == te["target"].values).all()

te["p_C"], te["pred_C"] = p_test, (p_test >= 0.5).astype(int)
te["p_A"], te["pred_A"] = P["score::HGB [Harga]"].values, P["pred::HGB [Harga]"].values
y = te["target"].values
benar_C, benar_A = te["pred_C"].values == y, te["pred_A"].values == y
te["benar_C"], te["benar_A"] = benar_C, benar_A
te["kategori"] = np.select([benar_C & benar_A, benar_C & ~benar_A, ~benar_C & benar_A],
                           ["keduanya benar", "hanya C benar", "hanya A benar"], "keduanya salah")
te["dP_rupiah"] = te["usd_idr"] * (np.exp(te["target_ret"]) - 1)   # gerak kurs besok dalam Rupiah

def mbb_ci(stat, n, B=2000, block=5, seed=SEED):
    g = np.random.default_rng(seed)
    nb = int(np.ceil(n / block))
    out = []
    for _ in range(B):
        ix = (g.integers(0, n - block + 1, nb)[:, None] + np.arange(block)).ravel()[:n]
        out.append(stat(ix))
    return [round(float(v), 3) for v in np.percentile(out, [2.5, 97.5])]

n = len(y)
tab = te["kategori"].value_counts().reindex(["keduanya benar", "hanya C benar", "hanya A benar", "keduanya salah"])
b, c = int(tab["hanya C benar"]), int(tab["hanya A benar"])
ringkas = {
    "DA_C": float(benar_C.mean()), "DA_C_ci95": mbb_ci(lambda ix: benar_C[ix].mean(), n),
    "DA_A": float(benar_A.mean()), "DA_A_ci95": mbb_ci(lambda ix: benar_A[ix].mean(), n),
    "dDA_C_minus_A": float(benar_C.mean() - benar_A.mean()),
    "dDA_ci95": mbb_ci(lambda ix: benar_C[ix].mean() - benar_A[ix].mean(), n),
    "mcnemar_p_dua_sisi": float(binomtest(b, b + c, 0.5).pvalue),
    "binom_p_C_vs_50": float(binomtest(int(benar_C.sum()), n, 0.5, alternative="greater").pvalue),
    "binom_p_C_vs_mayoritas": float(binomtest(int(benar_C.sum()), n, float(y.mean()), alternative="greater").pvalue),
    "porsi_tebak_naik_C": float(te["pred_C"].mean()), "porsi_tebak_naik_A": float(te["pred_A"].mean()),
    "porsi_naik_aktual": float(y.mean()),
    "porsi_naik_di_hanya_C_benar": float(y[te["kategori"].values == "hanya C benar"].mean()),
}
print(tab.to_string(), "\n")
for k, v in ringkas.items():
    print(f"{k:28s} {v}")

kategori
keduanya benar    57
hanya C benar     47
hanya A benar     32
keduanya salah    42 

DA_C                         0.5842696629213483
DA_C_ci95                    [0.506, 0.652]
DA_A                         0.5
DA_A_ci95                    [0.421, 0.573]
dDA_C_minus_A                0.0842696629213483
dDA_ci95                     [-0.006, 0.18]
mcnemar_p_dua_sisi           0.11466035574580133
binom_p_C_vs_50              0.014722987533758228
binom_p_C_vs_mayoritas       0.4713430650497415
porsi_tebak_naik_C           0.6573033707865169
porsi_tebak_naik_A           0.48314606741573035
porsi_naik_aktual            0.5786516853932584
porsi_naik_di_hanya_C_benar  0.7446808510638298


In [4]:
# DA menurut jumlah artikel per hari, besar gerakan kurs, dan bulan
te["kelompok_artikel"] = pd.cut(te["jumlah_berita"], [-1, 2, 7, 10**6], labels=["0-2 artikel", "3-7 artikel", ">=8 artikel"])
te["kelompok_gerak"] = np.where(te["dP_rupiah"].abs() <= 10, "|dP| <= Rp10", "|dP| > Rp10")
te["bulan"] = te["date"].dt.to_period("M").astype(str)
per_kelompok = pd.concat([
    te.groupby(k, observed=True).agg(hari=("target", "size"), DA_C=("benar_C", "mean"), DA_A=("benar_A", "mean"))
      .rename_axis("kelompok").assign(jenis=k)
    for k in ["kelompok_artikel", "kelompok_gerak", "bulan"]]).reset_index()
per_kelompok[["DA_C", "DA_A"]] = per_kelompok[["DA_C", "DA_A"]].round(3)
per_kelompok

,kelompok,hari,DA_C,DA_A,jenis
0,0-2 artikel,11,0.364,0.727,kelompok_artikel
1,3-7 artikel,23,0.565,0.478,kelompok_artikel
2,>=8 artikel,144,0.604,0.486,kelompok_artikel
3,|dP| <= Rp10,28,0.571,0.393,kelompok_gerak
4,|dP| > Rp10,150,0.587,0.520,kelompok_gerak
5,2025-11,3,1.000,1.000,bulan
6,2025-12,21,0.571,0.476,bulan
7,2026-01,20,0.600,0.400,bulan
8,2026-02,18,0.722,0.778,bulan
9,2026-03,17,0.529,0.471,bulan


In [5]:
# Tanggal di mana teks membantu (hanya C benar) dan menambah noise (hanya A benar), diurutkan dari gerak kurs terbesar
lab = pd.read_json(FEATURES_C / "c5_llm_labels.jsonl", lines=True)
lab["date"] = text.index[lab["pos"].values]
porsi_event = pd.crosstab(lab["date"], lab["event"], normalize="index").add_prefix("event_")
te = te.merge(porsi_event, left_on="date", right_index=True, how="left")
EV = list(porsi_event.columns)
te[EV] = te[EV].fillna(0.0)

def judul_awal(d, k=2):
    return " | ".join([x.strip() for x in str(text.loc[d, TITLE]).split(" | ") if x.strip()][:k])

contoh = []
for kat in ["hanya C benar", "hanya A benar"]:
    sub = te[te["kategori"] == kat].assign(abs_dP=lambda x: x["dP_rupiah"].abs()).nlargest(6, "abs_dP")
    for _, r in sub.iterrows():
        contoh.append({"kategori": kat, "date": r["date"].date(), "dP_rupiah": round(r["dP_rupiah"]), "target": r["target"],
                       "p_C": round(r["p_C"], 2), "p_A": round(r["p_A"], 2), "jumlah_berita": r["jumlah_berita"],
                       "event_dominan": r[EV].astype(float).idxmax().replace("event_", ""), "judul": judul_awal(r["date"])})
contoh = pd.DataFrame(contoh)
pd.set_option("display.max_colwidth", 120)
contoh

,kategori,date,dP_rupiah,target,p_C,p_A,jumlah_berita,event_dominan,judul
0,hanya C benar,2026-05-11,99,1,0.52,0.47,12,war/conflict,"Putin says he thinks Russia-Ukraine war is coming to an end | U.S., Iran no closer to ending war as Qatari tanker sa..."
1,hanya C benar,2026-07-16,-97,0,0.38,0.70,8,economic data,Dallas Fed President Logan calls for 'modestly' higher interest rates | Banking Committee's Sen. Rounds liked Warsh'...
2,hanya C benar,2026-02-27,69,1,0.64,0.34,7,monetary policy,Anthropic faces lose-lose scenario in Pentagon conflict as deadline for policy change looms | 10-year yield falls be...
3,hanya C benar,2026-06-02,68,1,0.52,0.33,28,monetary policy,Oil exports through the Strait of Hormuz might not return to levels seen before the Iran war | European defense stoc...
4,hanya C benar,2026-01-28,63,1,0.56,0.38,15,monetary policy,Gold jumps to fresh highs as Asia stock markets mostly rise after U.S. Fed holds rates | Where to still find solid y...
5,hanya C benar,2026-07-23,58,1,0.59,0.42,25,economic data,"U.S. to see higher generic drug prices thanks to tariffs, CEO of leading India pharma firm warns | 10-year Treasury ..."
6,hanya A benar,2026-06-12,-202,0,0.52,0.42,2,economic data,"Treasury yields move higher as traders monitor potential U.S.-Iran peace deal | For Warsh as Fed chair, silence may ..."
7,hanya A benar,2026-08-04,-100,0,0.66,0.45,16,monetary policy,"Iran, Oman say Strait of Hormuz talks in final stages, as Trump again teases a deal being close | Here's Todd Gordon..."
8,hanya A benar,2026-04-28,79,1,0.44,0.60,11,economic data,"Dollar firms, yen steady as markets weigh Iran war, split BOJ decision | Inflation could get in the way of Warsh's d..."
9,hanya A benar,2026-02-20,-67,0,0.53,0.46,16,monetary policy,Supreme Court Trump tariff decision impact: What to expect as fight for billions in refunds begins | Fourth-quarter ...


In [6]:
# Simpan hasil analisis error
kolom = ["date", "usd_idr", "target", "dP_rupiah", "jumlah_berita", "p_A", "pred_A", "p_C", "pred_C", "kategori"] + EV
te[kolom].to_csv(ANALYSIS / "error_per_tanggal.csv", index=False)
per_kelompok.to_csv(ANALYSIS / "error_per_kelompok.csv", index=False)
contoh.to_csv(ANALYSIS / "error_contoh_tanggal.csv", index=False)
(ANALYSIS / "error_ringkasan.json").write_text(json.dumps({"tabel": tab.to_dict(), **ringkas}, indent=2))
print("tersimpan di", ANALYSIS)

tersimpan di ../results_c/analysis


## 3. Feature importance (permutation importance)

Penurunan AUC test ketika satu fitur (30 ulangan) atau satu kelompok fitur (50 ulangan) diacak antarhari pada model final.
Dihitung setelah test dinilai, jadi hanya untuk interpretasi, tidak untuk memilih model.

In [7]:
X_te = D["test"][COLS].reset_index(drop=True)
auc_dasar = roc_auc_score(y, model_c.predict_proba(X_te)[:, 1])
rng = np.random.default_rng(SEED)

def turun_auc(cols, R):
    out = []
    for _ in range(R):
        Xp = X_te.copy()
        Xp[cols] = Xp[cols].values[rng.permutation(len(Xp))]
        out.append(auc_dasar - roc_auc_score(y, model_c.predict_proba(Xp)[:, 1]))
    return float(np.mean(out)), [round(float(v), 3) for v in np.percentile(out, [2.5, 97.5])]

print("AUC test dasar:", round(auc_dasar, 4))
imp_kelompok = {g: turun_auc(cols, 50) for g, cols in [("harga", PRICE), ("teks", TXT)]}
for g, (m, ci) in imp_kelompok.items():
    print(f"kelompok {g:5s}: turun AUC {m:.3f}  CI95 {ci}")
imp_fitur = pd.Series({c: turun_auc([c], 30)[0] for c in COLS}, name="turun_auc").sort_values(ascending=False)
imp_fitur.head(10).round(4)

AUC test dasar: 0.5937


kelompok harga: turun AUC 0.020  CI95 [-0.016, 0.054]
kelompok teks : turun AUC 0.069  CI95 [0.003, 0.137]


c2_art_cls_3     0.0405
c2_art_cls_4     0.0200
c2_art_cls_9     0.0176
ret_lag1         0.0128
c2_art_cls_7     0.0124
ret_lag2         0.0108
c2_art_cls_1     0.0083
c2_art_cls_8     0.0078
c2_art_cls_16    0.0030
c2_art_cls_11    0.0018
Name: turun_auc, dtype: float64

In [8]:
# Tafsir komponen PCA teratas: judul hari train dengan skor tertinggi/terendah dan korelasi dengan porsi label peristiwa C5 (train)
tr_dates = D["train"]["date"]
ev_train = pd.crosstab(lab["date"], lab["event"], normalize="index").reindex(tr_dates).fillna(0.0)
tafsir = []
for c_ in [c for c in imp_fitur.index if c.startswith("c2_art_cls_") and c != "c2_art_cls_has"][:3]:
    v = comp.loc[tr_dates, c_]
    rho = {e: spearmanr(v.values, ev_train[e].values)[0] for e in ev_train.columns}
    top = sorted(rho.items(), key=lambda t: -abs(t[1]))[:3]
    tafsir.append({"komponen": c_,
                   "judul_tertinggi": " || ".join(judul_awal(d, 1) for d in v.nlargest(3).index),
                   "judul_terendah": " || ".join(judul_awal(d, 1) for d in v.nsmallest(3).index),
                   "rho_event_terkuat": ", ".join(f"{e} {r:+.2f}" for e, r in top)})
tafsir = pd.DataFrame(tafsir)

# Apakah perubahan prediksi (p_C - p_A) terkait jenis peristiwa tertentu di test?
geser = te["p_C"] - te["p_A"]
kor_event = pd.DataFrame([{"event": e.replace("event_", ""), "rho": spearmanr(geser, te[e])[0], "p": spearmanr(geser, te[e])[1]} for e in EV]).round(3)
print(kor_event.to_string(index=False))
tafsir

             event    rho     p
     economic data -0.032 0.670
elections/politics -0.017 0.821
 financial markets  0.039 0.606
   monetary policy  0.009 0.903
        oil/energy -0.003 0.965
             other -0.041 0.583
     trade/tariffs -0.087 0.250
      war/conflict  0.065 0.388


,komponen,judul_tertinggi,judul_terendah,rho_event_terkuat
0,c2_art_cls_3,Oil extends losses on prospect of higher OPEC+ supply || Oil prices climb on hurricane impact ahead of U.S. rate dec...,How 5 companies are using pricing power to protect profits during high inflation || There are key differences betwee...,"financial markets +0.18, oil/energy +0.15, trade/tariffs -0.11"
1,c2_art_cls_4,"China has spent at least $230 billion to build its EV industry, new study finds || Linde's ability to manage inflati...","From mortgages to auto loans, experts weigh in on when — or if — to refinance as interest rates fall || Watch Jerome...","monetary policy -0.19, financial markets +0.15, war/conflict +0.09"
2,c2_art_cls_9,Here are the top-performing technology stocks of 2021 || Competing bills could ban lawmakers from trading stocks whi...,"In the face of trade tensions, China says it will focus on its own economy || Huawei's smartphone spinoff Honor plan...","economic data -0.21, oil/energy +0.16, war/conflict -0.16"


In [9]:
imp_fitur.to_csv(ANALYSIS / "permutation_importance_fitur.csv")
pd.DataFrame([{"kelompok": g, "turun_auc": m, "ci95_bawah": ci[0], "ci95_atas": ci[1]} for g, (m, ci) in imp_kelompok.items()]) \
  .to_csv(ANALYSIS / "permutation_importance_kelompok.csv", index=False)
tafsir.to_csv(ANALYSIS / "tafsir_komponen_pca.csv", index=False)
kor_event.to_csv(ANALYSIS / "korelasi_geser_prediksi_event.csv", index=False)
print("tersimpan di", ANALYSIS)

tersimpan di ../results_c/analysis


## 4. Ringkasan

- Model C benar pada 47 hari saat Model A salah, sebaliknya 32 hari. Selisihnya belum signifikan (McNemar p ≈ 0,115; CI selisih DA memuat nol).
- DA Model C signifikan di atas 50% tetapi tidak di atas proporsi kelas mayoritas test (0,579). Model C lebih sering menebak NAIK (≈66% hari).
- Hari dengan sangat sedikit artikel (0-2) adalah titik lemah Model C; pada hari dengan banyak artikel Model C lebih baik dari A.
- Mengacak fitur teks menurunkan AUC test lebih banyak daripada mengacak fitur harga, tetapi tidak ada satu jenis peristiwa (label C5) yang berkorelasi jelas dengan perubahan prediksi.
- Visualisasi ruang embedding (t-SNE volatilitas tinggi vs rendah) ada di `07b_tsne_volatilitas.ipynb`; grafik ringkas analisis ini ada di `notebook/hasil_analisis_tugas3.ipynb`.